# Feature Engineering for Transformer

## Overview
This notebook demonstrates advanced feature engineering using Transformer architectures. We'll explore self-attention mechanisms, positional encoding, and multi-head attention for extracting high-quality features from sequential data.

## Concepts
- **Self-Attention Mechanisms**: Understanding how transformers capture long-range dependencies
- **Multi-Head Attention**: Parallel attention computations for diverse feature extraction
- **Positional Encoding**: Injecting sequence order information
- **Transformer Encoder**: Building blocks for feature extraction
- **Feature Visualization**: Understanding what transformers learn
- **Practical Applications**: Real-world use cases and implementation tips

---

In [ ]:
# Essential Imports and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Dataset, TensorDataset
import math
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Device configuration with fallback
def setup_device():
    """Setup device with comprehensive error handling"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"CUDA available: {torch.cuda.get_device_name()}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        device = torch.device('cpu')
        print("⚠️ CUDA not available, using CPU")
        print("For better performance, ensure CUDA is properly installed")
    
    return device

device = setup_device()

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print(f"🔧 PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"Notebook: Transformer Feature Engineering")
print("=" * 60)

In [ ]:
# Data Preparation for Transformer Training
def generate_synthetic_sequence_data(num_samples=6000, seq_length=32):
    """Generate synthetic sequential data for transformer training"""
    
    # Create diverse sequence patterns
    patterns = {
        'ascending': lambda: sorted(np.random.randint(1, 100, seq_length)),
        'descending': lambda: sorted(np.random.randint(1, 100, seq_length), reverse=True),
        'periodic': lambda: [int(50 + 30*np.sin(2*np.pi*i/8)) for i in range(seq_length)],
        'random': lambda: np.random.randint(1, 100, seq_length).tolist(),
        'spike': lambda: [50]*seq_length[:seq_length//2] + [90]*seq_length[seq_length//2:],
    }
    
    sequences = []
    labels = []
    
    for i in range(num_samples):
        # Choose pattern type
        pattern_type = np.random.choice(list(patterns.keys()))
        sequence = patterns[pattern_type]()
        
        # Add some noise
        sequence = [max(1, min(99, x + np.random.randint(-5, 6))) for x in sequence]
        
        sequences.append(sequence)
        # Create classification labels based on pattern characteristics
        if pattern_type in ['ascending', 'spike']:
            labels.append(1)  # Positive trend
        else:
            labels.append(0)  # Non-positive trend
    
    return sequences, labels

# Generate training data
print("Generating synthetic sequence data for Transformers...")
sequences, labels = generate_synthetic_sequence_data(num_samples=6000, seq_length=32)

print(f"Generated {len(sequences)} sequences")
print(f"Sequence length: {len(sequences[0])}")
print(f"Positive samples: {sum(labels)}")
print(f"Negative samples: {len(labels) - sum(labels)}")

# Display sample sequences
print("\n📊 Sample sequences:")
for i in range(3):
    trend = "Positive" if labels[i] == 1 else "Negative"
    print(f"{i+1}. [{trend}] {sequences[i][:10]}... (first 10 elements)")

# Convert to tensors and create data loaders
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Convert to tensors
X = torch.FloatTensor(sequences).unsqueeze(-1)  # Add feature dimension
y = torch.LongTensor(labels)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData preparation complete:")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Input shape: {X_train.shape}")
print(f"Batch size: {batch_size}")

# Test data loader
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"Sequences: {sample_batch[0].shape}")  # [batch_size, seq_length, input_dim]
print(f"Labels: {sample_batch[1].shape}")     # [batch_size]

In [ ]:
# Positional Encoding for Transformers
class PositionalEncoding(nn.Module):
    """Positional encoding as described in 'Attention Is All You Need'"""
    
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Create div_term for sine and cosine
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices  
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension and register as buffer
        pe = pe.unsqueeze(0).transpose(0, 1)  # [max_len, 1, d_model]
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        """Add positional encoding to input embeddings"""
        # x shape: [seq_len, batch_size, d_model]
        return x + self.pe[:x.size(0), :]

# Multi-Head Self-Attention
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention mechanism"""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadSelfAttention, self).__init__()
        
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Compute scaled dot-product attention"""
        # Q, K, V shape: [batch_size, num_heads, seq_len, d_k]
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        context = torch.matmul(attention_weights, V)
        
        return context, attention_weights
    
    def forward(self, x, mask=None, return_attention=False):
        batch_size, seq_len, d_model = x.size()
        
        # Linear transformations and split into heads
        Q = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply scaled dot-product attention
        context, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and put through final linear layer
        context = context.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model
        )
        output = self.w_o(context)
        
        if return_attention:
            return output, attention_weights
        
        return output

# Transformer Encoder Layer
class TransformerEncoderLayer(nn.Module):
    """Single transformer encoder layer"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderLayer, self).__init__()
        
        self.self_attention = MultiHeadSelfAttention(d_model, num_heads, dropout)
        
        # Feed-forward network
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None, return_attention=False):
        # Self-attention with residual connection and layer norm
        if return_attention:
            attn_output, attention_weights = self.self_attention(x, mask, return_attention=True)
        else:
            attn_output = self.self_attention(x, mask)
            attention_weights = None
            
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection and layer norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        if return_attention:
            return x, attention_weights
        
        return x

print("Transformer components implemented:")
print("- PositionalEncoding")
print("- MultiHeadSelfAttention")
print("- TransformerEncoderLayer")

In [ ]:
# Complete Transformer Feature Extractor
class TransformerFeatureExtractor(nn.Module):
    """Complete Transformer model for feature extraction"""
    
    def __init__(self, input_dim=1, d_model=128, num_heads=8, num_layers=6, 
                 d_ff=512, max_seq_length=1000, num_classes=2, dropout=0.1):
        super(TransformerFeatureExtractor, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = num_layers
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_seq_length)
        
        # Transformer encoder layers
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Feature extraction
        self.feature_extractor = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Global pooling options
        self.pooling_type = 'mean'  # Options: 'mean', 'max', 'cls', 'attention'
        
        # Classification head
        self.classifier = nn.Linear(d_model // 4, num_classes)
        
        # Attention pooling (if using attention pooling)
        self.attention_pool = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, return_attention=False, return_features=False):
        # x shape: [batch_size, seq_length, input_dim]
        batch_size, seq_length, _ = x.size()
        
        # Project input to model dimension
        x = self.input_projection(x)  # [batch_size, seq_length, d_model]
        x = x * math.sqrt(self.d_model)  # Scale embeddings
        
        # Add positional encoding
        x = x.transpose(0, 1)  # [seq_length, batch_size, d_model]
        x = self.pos_encoding(x)
        x = x.transpose(0, 1)  # [batch_size, seq_length, d_model]
        
        x = self.dropout(x)
        
        # Pass through transformer encoder layers
        attention_weights = []
        for i, layer in enumerate(self.encoder_layers):
            if return_attention and i == len(self.encoder_layers) - 1:
                # Get attention from last layer
                x, attn = layer(x, return_attention=True)
                attention_weights = attn
            else:
                x = layer(x)
        
        # Global pooling to get sequence representation
        if self.pooling_type == 'mean':
            # Average pooling
            pooled = torch.mean(x, dim=1)  # [batch_size, d_model]
        elif self.pooling_type == 'max':
            # Max pooling
            pooled, _ = torch.max(x, dim=1)  # [batch_size, d_model]
        elif self.pooling_type == 'attention':
            # Attention-based pooling
            attention_scores = self.attention_pool(x)  # [batch_size, seq_length, 1]
            attention_weights_pool = F.softmax(attention_scores, dim=1)
            pooled = torch.sum(x * attention_weights_pool, dim=1)  # [batch_size, d_model]
        else:
            # Use first token (CLS-like)
            pooled = x[:, 0, :]  # [batch_size, d_model]
        
        # Extract features
        features = self.feature_extractor(pooled)
        
        if return_features:
            return features
        
        # Classification
        logits = self.classifier(features)
        
        if return_attention and len(attention_weights) > 0:
            return logits, features, attention_weights
        
        return logits, features

# Vision Transformer-style Feature Extractor (alternative implementation)
class ViTStyleFeatureExtractor(nn.Module):
    """Vision Transformer style feature extractor for sequences"""
    
    def __init__(self, seq_length=32, input_dim=1, patch_size=4, d_model=128, 
                 num_heads=8, num_layers=6, num_classes=2, dropout=0.1):
        super(ViTStyleFeatureExtractor, self).__init__()
        
        self.patch_size = patch_size
        self.num_patches = seq_length // patch_size
        self.d_model = d_model
        
        # Patch embedding
        self.patch_embedding = nn.Linear(patch_size * input_dim, d_model)
        
        # Class token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        # Position embeddings
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Feature head
        self.feature_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Linear(d_model // 2, num_classes)
        
    def forward(self, x, return_features=False):
        batch_size, seq_length, input_dim = x.size()
        
        # Create patches
        x = x.reshape(batch_size, self.num_patches, self.patch_size * input_dim)
        
        # Patch embedding
        x = self.patch_embedding(x)  # [batch_size, num_patches, d_model]
        
        # Add class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)  # [batch_size, num_patches+1, d_model]
        
        # Add position embeddings
        x = x + self.pos_embedding
        
        # Transformer encoding
        x = self.transformer(x)  # [batch_size, num_patches+1, d_model]
        
        # Use class token for classification
        cls_output = x[:, 0]  # [batch_size, d_model]
        
        # Extract features
        features = self.feature_head(cls_output)
        
        if return_features:
            return features
            
        # Classification
        logits = self.classifier(features)
        
        return logits, features

# Initialize models
print("Creating Transformer models...")

# Standard Transformer
transformer_model = TransformerFeatureExtractor(
    input_dim=1,
    d_model=128,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    max_seq_length=64,
    num_classes=2,
    dropout=0.1
).to(device)

# ViT-style model
vit_model = ViTStyleFeatureExtractor(
    seq_length=32,
    input_dim=1,
    patch_size=4,
    d_model=128,
    num_heads=8,
    num_layers=6,
    num_classes=2,
    dropout=0.1
).to(device)

print(f"Models created on {device}")
print(f"Standard Transformer parameters: {sum(p.numel() for p in transformer_model.parameters()):,}")
print(f"ViT-style Transformer parameters: {sum(p.numel() for p in vit_model.parameters()):,}")

# Test forward pass
with torch.no_grad():
    sample_input = sample_batch[0].to(device)[:4]  # First 4 samples
    
    # Test standard transformer
    output1, features1, attention1 = transformer_model(sample_input, return_attention=True)
    print(f"\nStandard Transformer test:")
    print(f"Input shape: {sample_input.shape}")
    print(f"Output shape: {output1.shape}")
    print(f"Features shape: {features1.shape}")
    print(f"Attention shape: {attention1.shape}")
    
    # Test ViT-style
    output2, features2 = vit_model(sample_input)
    print(f"\nViT-style Transformer test:")
    print(f"Output shape: {output2.shape}")
    print(f"Features shape: {features2.shape}")

In [ ]:
# Training and Analysis Functions for Transformers
def train_transformer_model(model, train_loader, test_loader, num_epochs=15, lr=0.0001):
    """Train transformer model with learning rate scheduling"""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr*10, epochs=num_epochs, 
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    
    train_losses = []
    test_losses = []
    train_accuracies = []
    test_accuracies = []
    
    print("Starting Transformer training...")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, targets) in enumerate(train_loader):
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
        
        # Test phase
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for data, targets in test_loader:
                data, targets = data.to(device), targets.to(device)
                outputs, _ = model(data)
                loss = criterion(outputs, targets)
                
                test_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                test_total += targets.size(0)
                test_correct += (predicted == targets).sum().item()
        
        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_test_loss = test_loss / len(test_loader)
        epoch_train_acc = 100. * train_correct / train_total
        epoch_test_acc = 100. * test_correct / test_total
        
        train_losses.append(epoch_train_loss)
        test_losses.append(epoch_test_loss)
        train_accuracies.append(epoch_train_acc)
        test_accuracies.append(epoch_test_acc)
        
        if (epoch + 1) % 3 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}]')
            print(f'  Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
            print(f'  Test Loss: {epoch_test_loss:.4f}, Test Acc: {epoch_test_acc:.2f}%')
            print(f'  LR: {scheduler.get_last_lr()[0]:.6f}')
    
    return {
        'train_losses': train_losses,
        'test_losses': test_losses,
        'train_accuracies': train_accuracies,
        'test_accuracies': test_accuracies
    }

def visualize_attention_patterns(model, data_loader, num_samples=3):
    """Visualize multi-head attention patterns"""
    model.eval()
    
    # Get sample data
    data_iter = iter(data_loader)
    data, targets = next(data_iter)
    data, targets = data.to(device), targets.to(device)
    
    with torch.no_grad():
        if hasattr(model, 'transformer'):
            # ViT-style model
            outputs, features = model(data[:num_samples])
            attention_weights = None
            model_name = "ViT-style"
        else:
            # Standard transformer
            outputs, features, attention_weights = model(data[:num_samples], return_attention=True)
            model_name = "Standard Transformer"
    
    if attention_weights is not None:
        # Visualize attention patterns
        attention_weights = attention_weights.cpu().numpy()
        batch_size, num_heads, seq_len, _ = attention_weights.shape
        
        fig, axes = plt.subplots(num_samples, num_heads, figsize=(3*num_heads, 3*num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)
        
        for sample in range(num_samples):
            for head in range(num_heads):
                ax = axes[sample, head] if num_samples > 1 else axes[head]
                
                # Plot attention matrix
                im = ax.imshow(attention_weights[sample, head], cmap='Blues', aspect='auto')
                ax.set_title(f'Sample {sample+1}, Head {head+1}')
                ax.set_xlabel('Key Position')
                ax.set_ylabel('Query Position')
                
                # Add colorbar
                plt.colorbar(im, ax=ax, shrink=0.8)
        
        plt.suptitle(f'{model_name} - Multi-Head Attention Patterns')
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  {model_name} doesn't return attention weights for visualization")

def analyze_transformer_features(model, data_loader, model_name):
    """Extract and analyze transformer features"""
    model.eval()
    all_features = []
    all_labels = []
    
    print(f"Extracting features from {model_name}...")
    
    with torch.no_grad():
        for data, targets in data_loader:
            data, targets = data.to(device), targets.to(device)
            features = model(data, return_features=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(targets.cpu().numpy())
    
    features_array = np.concatenate(all_features, axis=0)
    labels_array = np.concatenate(all_labels, axis=0)
    
    print(f"Extracted features shape: {features_array.shape}")
    
    # Feature analysis
    print(f"\n{model_name} Feature Analysis")
    print("-" * 50)
    print(f"Feature dimensions: {features_array.shape}")
    print(f"Feature mean: {features_array.mean():.4f}")
    print(f"Feature std: {features_array.std():.4f}")
    print(f"Feature range: [{features_array.min():.4f}, {features_array.max():.4f}]")
    
    # PCA Analysis
    from sklearn.decomposition import PCA
    pca = PCA(n_components=min(50, features_array.shape[1]))
    features_pca = pca.fit_transform(features_array)
    
    print(f"PCA explained variance (top 10): {pca.explained_variance_ratio_[:10].round(3)}")
    print(f"Cumulative variance explained: {pca.explained_variance_ratio_.sum():.3f}")
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Feature distribution
    axes[0, 0].hist(features_array.flatten(), bins=50, alpha=0.7, color='lightcoral')
    axes[0, 0].set_title(f'{model_name} - Feature Distribution')
    axes[0, 0].set_xlabel('Feature Value')
    axes[0, 0].set_ylabel('Frequency')
    
    # PCA variance
    axes[0, 1].plot(pca.explained_variance_ratio_[:min(20, len(pca.explained_variance_ratio_))], 'o-')
    axes[0, 1].set_title(f'{model_name} - PCA Components')
    axes[0, 1].set_xlabel('Component')
    axes[0, 1].set_ylabel('Explained Variance Ratio')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 2D visualization
    pca_2d = PCA(n_components=2)
    features_2d = pca_2d.fit_transform(features_array)
    
    scatter = axes[1, 0].scatter(features_2d[:, 0], features_2d[:, 1], 
                                c=labels_array, cmap='viridis', alpha=0.6, s=20)
    axes[1, 0].set_title(f'{model_name} - 2D Feature Space')
    axes[1, 0].set_xlabel('First Principal Component')
    axes[1, 0].set_ylabel('Second Principal Component')
    plt.colorbar(scatter, ax=axes[1, 0])
    
    # Feature importance (using first 16 features)
    feature_importance = np.abs(features_array).mean(axis=0)[:16]
    axes[1, 1].bar(range(len(feature_importance)), feature_importance)
    axes[1, 1].set_title(f'{model_name} - Feature Importance (First 16)')
    axes[1, 1].set_xlabel('Feature Index')
    axes[1, 1].set_ylabel('Mean Absolute Value')
    
    plt.tight_layout()
    plt.show()
    
    return features_array, labels_array

print("✅ Transformer training and analysis functions defined")

# Feature Engineering with PyTorch for Transformer

## Self-Attention Based Feature Learning

This notebook demonstrates **Transformer** architectures for feature engineering, covering:

- **Multi-Head Self-Attention** mechanisms
- **Positional Encoding** for sequence understanding
- **Transformer Encoder** blocks
- **Feature extraction** from attention patterns
- **Vision Transformers (ViT)** for image features
- **Attention visualization** and interpretability

### Learning Objectives:
1. Understand the Transformer architecture components
2. Implement self-attention mechanisms from scratch
3. Build complete Transformer models for feature extraction
4. Visualize attention patterns for interpretability
5. Apply Transformers to both text and image data

### 🔧 Key Components:
- **Self-Attention**: Learn relationships between all positions
- **Multi-Head Attention**: Multiple attention perspectives
- **Feed-Forward Networks**: Position-wise transformations
- **Layer Normalization**: Training stability
- **Positional Encoding**: Sequence position information

---